# Dataset

> Frozen test records and teacher-labelled training records, stored as versioned JSON Lines files

In [ ]:
#| default_exp dataset

The dataset is the lasting artifact of the [scope](design/scope.ipynb): every model is rebuilt from a version of it. Each version is a JSON Lines file under `data/`, committed with the code that made it, and never overwritten. A training file holds one line for each record and teacher. The test file holds the records only, because the code checks score a message against its record.

In [ ]:
#| export
import json, asyncio
from pathlib import Path
from edgeslm.synth import *
from edgeslm.coffee import *
from edgeslm.checks import *
from edgeslm.teacher import *

In [ ]:
#| hide
from collections import Counter
from fastcore.test import *

## Files

`write_jsonl` and `read_jsonl` write and read one JSON object per line, keeping non-ASCII text such as farm names and emoji readable:

In [ ]:
#| export
def write_jsonl(
    path, # File to write; refused if it exists, since dataset versions are never overwritten
    rows:list, # JSON-serialisable dicts
):
    "Write `rows` to a new JSON Lines file"
    path = Path(path)
    if path.exists(): raise FileExistsError(f'{path} exists: write a new version instead')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows))

def read_jsonl(path)->list:
    "The rows of a JSON Lines file"
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l]

## Records

`records` samples `n` records with `sample` and gives each an id from its seed and position, such as `s4-0017`. Records also found in `exclude`, such as the test set, are dropped, so no test input is ever used for training:

In [ ]:
#| export
def _key(rec): return json.dumps(rec, sort_keys=True)

def records(
    n:int, # Number of records to sample
    seed:int, # Seed of the sample
    exclude:list=(), # Rows whose records must not appear
)->list: # Dicts with `id`, `tuple` and `record`
    "`n` sampled records with ids, minus any in `exclude`"
    skip = {_key(r['record']) for r in exclude}
    rows = [dict(id=f's{seed}-{i:04d}', tuple=s['tuple'], record=s['input']) for i, s in enumerate(sample(DIMS, triage, n, seed=seed))]
    return [r for r in rows if _key(r['record']) not in skip]

## The frozen test set

The test set used to be regenerated from seed 1, so every change to the generator changed it. It is now written once, and read from its file from then on. This cell made it, and does not run in tests:

In [ ]:
#| eval: false
write_jsonl('../data/test-v1.jsonl', records(270, seed=1))

It holds 10 records for each of the 27 tuples:

In [ ]:
test = read_jsonl('../data/test-v1.jsonl')
test_eq(set(Counter(tuple(r['tuple'].values()) for r in test).values()), {10})
len(test), test[0]['id']

(270, 's1-0000')

## Teacher messages

`label` has a teacher write the message for one record and runs the code checks on it. A message that fails is written again once, with the failed checks' failure modes added to the prompt. The row keeps the second attempt and its checks either way, so that failures can be counted and left out of training. `label_all` runs every record through every teacher, a few calls at a time to stay within rate limits:

In [ ]:
#| export
def _note(res):
    "The prompt note for a message whose checks `res` failed"
    fails = [f for f in FAILURES if any(not res[c] for c in f['checks'])]
    return 'A previous message for this record made these mistakes: ' + ' '.join(f"{f['what']} Fix: {f['fix']}" for f in fails)

async def label(
    row:dict, # A row from `records`
    model:str=MODEL, # Teacher model
)->dict: # `row` with the message, its checks, the attempt number and the provenance
    "Have `model` write and check the message for `row`"
    out = await teach(row['record'], model)
    res, attempt = check(row['record'], out['message']), 1
    if not all(res.values()):
        out = await teach(row['record'], model, note=_note(res))
        res, attempt = check(row['record'], out['message']), 2
    return dict(row, message=out.pop('message'), checks=res, attempt=attempt, **out)

async def label_all(
    rows:list, # Rows from `records`
    models:list, # Teacher models
    n_workers:int=8, # Calls at a time
)->list: # One labelled row per record and model
    "Label every row with every model"
    sem = asyncio.Semaphore(n_workers)
    async def _one(r, m):
        async with sem: return await label(r, m)
    return await asyncio.gather(*[_one(r, m) for m in models for r in rows])

This cell wrote the first training set: 200 records from seed 4, each labelled by Haiku and Sonnet. It needs `ANTHROPIC_API_KEY` from the untracked `.env` file, and does not run in tests:

In [ ]:
#| eval: false
from dotenv import load_dotenv
load_dotenv()
rows = records(200, seed=4, exclude=test)
write_jsonl('../data/train-v1.jsonl', await label_all(rows, ['claude-haiku-4-5', 'claude-sonnet-4-6']))

Each record has one row per teacher, and every row was written with the same prompt:

In [ ]:
train = read_jsonl('../data/train-v1.jsonl')
test_eq(set(Counter(r['id'] for r in train).values()), {2})
test_eq(len({r['prompt'] for r in train}), 1)
len(train), train[0]['prompt']

(400, '3ec23e301454')

Almost every message passed the checks at the first attempt. Each teacher has one message that is still too long after the second attempt, and is left out of training:

In [ ]:
Counter((r['model'], r['attempt'], all(r['checks'].values())) for r in train)

Counter({('claude-haiku-4-5', 1, True): 199,
         ('claude-sonnet-4-6', 1, True): 198,
         ('claude-haiku-4-5', 2, False): 1,
         ('claude-sonnet-4-6', 2, True): 1,
         ('claude-sonnet-4-6', 2, False): 1})

The two teachers often write the same message. The rules and examples leave little room for wording, so the comparison of teachers in the scope measures less than planned:

In [ ]:
msgs = {(r['id'], r['model']): r['message'].strip() for r in train}
ids = sorted({r['id'] for r in train})
sum(msgs[i, 'claude-haiku-4-5'] == msgs[i, 'claude-sonnet-4-6'] for i in ids), len(ids)

(73, 200)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()